# 🩺 Pneumonia Detection — Colab runner
Run the cells top to bottom (Runtime ▸ Change runtime type ▸ **GPU** first).

1. Setup + install  2. *(optional)* Kaggle credentials  3. Train (downloads the dataset from Kaggle)  4. Launch the Streamlit app

> Research/education demo only — not a medical device.

In [ ]:
import glob, os, subprocess

# Move into the project folder (works wherever you unzipped it under /content)
if not os.path.exists("train.py"):
    hits = glob.glob("/content/**/train.py", recursive=True)
    assert hits, "Unzip the project under /content first (see README)."
    os.chdir(os.path.dirname(hits[0]))
print("Project folder:", os.getcwd())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU found (training will be slow)")

In [ ]:
# Colab already has TensorFlow / scikit-learn / matplotlib; we only add what is missing.
!pip -q install streamlit kagglehub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# OPTIONAL: Kaggle credentials from Colab Secrets (🔑 icon in the left bar).
# Public datasets normally download without this; only needed if you hit an auth error.
import os
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception as e:
    print("No Kaggle secrets set - trying an anonymous download.", type(e).__name__)

In [ ]:
# Downloads the dataset from Kaggle, trains (head -> fine-tune), evaluates on the test split,
# and writes pneumonia_detector_model.keras + artifacts/ next to app.py.
!python -u train.py

In [ ]:
# Quick look at the training results
import json
from IPython.display import Image, display
print(json.dumps(json.load(open("artifacts/metrics.json"))["test"], indent=2))
for f in ["training_curves.png", "confusion_matrix.png", "gradcam_samples.png"]:
    display(Image(f"artifacts/{f}"))

In [ ]:
# Launch Streamlit and expose it through a free Cloudflare quick tunnel (no account needed).
import re, subprocess, time
from IPython.display import HTML, display

subprocess.run("pkill -f 'streamlit run'; pkill -f cloudflared", shell=True)
st_log = open("streamlit.log", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--browser.gatherUsageStats", "false",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],  # needed behind the tunnel
    stdout=st_log, stderr=st_log)
time.sleep(8)

cf_log = open("cloudflared.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=cf_log, stderr=cf_log)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0)
        break
if url:
    display(HTML(f'<h3>✅ Your app: <a href="{url}" target="_blank">{url}</a></h3>'
                 '<p>Keep this Colab runtime running while you use it. Re-run this cell for a fresh link.</p>'))
else:
    print("Tunnel did not start - see cloudflared.log / streamlit.log:")
    print(open("cloudflared.log").read()[-1500:])

### Troubleshooting
- **Kaggle auth error** → add `KAGGLE_USERNAME` / `KAGGLE_KEY` Colab Secrets and re-run the cell above.
- **Link doesn't load** → re-run the last cell (quick-tunnel links are temporary), check `streamlit.log`.
- **Already have a trained model?** Put it next to `app.py` as `pneumonia_detector_model.keras`, skip the training cell, and run only the last cell.